# Day 19 — Missing Values & Duplicates
### Python for Data Science · Module 1 · Topic 1.18

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | **Finding what is missing** — and the gaps `isna()` cannot see | 20 min |
| 2 | Dropping | 15 min |
| 3 | **Filling — and the cost of each choice** | 25 min |
| 4 | Duplicates: exact, partial, fuzzy | 20 min |
| 5 | Mini build: a pipeline that reports itself | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **There is no neutral option.** Dropping rows loses information and can bias what remains.
> Filling invents numbers that were never measured. Leaving gaps makes every later step
> harder. All three are legitimate; none is free.
>
> The professional skill is not picking the right one — it is **stating which you picked,
> why, and what it cost.**

In [ ]:
import pandas as pd
import numpy as np
print("pandas", pd.__version__)

---
# 1. Finding what is missing

## 1.1 Three commands, on every new dataset

In [ ]:
df = pd.DataFrame({
    "a": [1, 2, np.nan, 4, 5],
    "b": [np.nan, np.nan, np.nan, 4.0, 5.0],
    "c": [1, 2, 3, 4, 5],
})

print("count per column:")
print(df.isna().sum())
print()
print("PROPORTION per column:")
print(df.isna().mean().round(2))
print()
print("rows with any gap:", df.isna().any(axis=1).sum(), "of", len(df))

**Read the proportion, not just the count.** Five missing out of ten is a different problem
from five out of ten thousand.

A column that is 60% empty (like `b` above) rarely deserves to be filled — doing so would
mean inventing more data than you measured.

## 1.2 Is it missing at random?

In [ ]:
# Before deciding what to do, ask WHY the values are missing.
staff = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Dev"],
    "dept":   ["Sales", "Exec", "Sales", "Exec", "Sales", "Exec"],
    "salary": [52000, np.nan, 48000, np.nan, 51000, np.nan],
})

print("departments WITH a salary recorded:")
print(staff[staff["salary"].notna()]["dept"].value_counts())
print()
print("departments MISSING a salary:")
print(staff[staff["salary"].isna()]["dept"].value_counts())

**The gaps are not random.** Every executive salary is missing. Dropping those rows removes
the highest earners, and the reported average falls — the missingness itself carries
information.

If high earners skip the salary question, your "average salary" after dropping is not an
average of your staff. It is an average of the people who answered.

## 1.3 ⚠️ The gaps `isna()` cannot see

In [ ]:
import io

csv = (
    "name,age,score,city\n"
    "Ravi,34,88,Pune\n"
    "Sara,-999,91,N/A\n"
    "Amit,29,,Delhi\n"
    "Neha,41,unknown,Pune\n"
)

raw = pd.read_csv(io.StringIO(csv))
print(raw)
print()
print("isna() found:", raw.isna().sum().to_dict())
print("  -> ONE of the four. -999, N/A and unknown are just data to pandas.")
print()
print("and look at the dtypes:")
print(raw.dtypes)

In [ ]:
# The fix, at read time
clean = pd.read_csv(io.StringIO(csv),
                    na_values=["-999", "N/A", "unknown", "", "-", "missing"])
print(clean)
print()
print("isna() now finds:", clean.isna().sum().to_dict())
print()
print("and the score column is numeric again:")
print(clean.dtypes)

### How to spot them before they bite

```python
df["city"].value_counts()    # look for odd entries
df["age"].describe()         # a min of -999 is a flag
df.dtypes                    # text where you expected numbers
```

**A numeric column stored as text almost always means a sentinel value is hiding in it.**
A minimum of −999, or a maximum of 9999, is a placeholder somebody chose decades ago.

In [ ]:
print(raw["age"].describe()[["min", "max"]])     # min of -999 is the giveaway
print()
print(raw["score"].value_counts())               # 'unknown' sitting among the numbers

---
# 2. Dropping

| Call | Removes | Use it when |
|---|---|---|
| `df.dropna()` | any row with **any** gap | rarely — the bluntest option |
| `df.dropna(how="all")` | only completely empty rows | always safe; a good first step |
| `df.dropna(subset=["salary"])` | rows missing **that** column | the usual choice |
| `df.dropna(thresh=3)` | rows with fewer than 3 real values | when rows need minimum evidence |
| `df.dropna(axis=1)` | whole **columns** with any gap | when a column is mostly empty |

In [ ]:
print("original rows      :", len(df))
print("dropna()           :", len(df.dropna()), "  <- one gap removes the whole row")
print("dropna(how='all')  :", len(df.dropna(how="all")))
print("dropna(thresh=2)   :", len(df.dropna(thresh=2)))
print("dropna(subset=['a']):", len(df.dropna(subset=["a"])))
print("dropna(axis=1) cols:", df.dropna(axis=1).columns.tolist())

> ### ⚠️ `dropna()` is greedier than it looks
> Five rows, one gap in column `a` and three in column `b` — and a bare `dropna()` leaves
> **two**. A single gap anywhere removes the whole row, including the columns that were
> perfectly fine.
>
> `dropna(subset=[...])` is almost always the better call.

---
# 3. Filling — and the cost

| Call | Puts in | Right for |
|---|---|---|
| `fillna(0)` | zero | counts, where missing truly means none |
| `fillna(s.mean())` | the average | roughly symmetric numeric data |
| `fillna(s.median())` | the middle value | skewed data, or data with outliers |
| `s.ffill()` | the previous value | ordered data: sensor readings, prices |
| `s.interpolate()` | a value on the line between | smooth numeric series over time |

In [ ]:
s = pd.Series([10., 20., np.nan, 40., 50.])

print("fillna(0)    :", s.fillna(0).tolist())
print("fillna(mean) :", s.fillna(s.mean()).tolist())
print("ffill()      :", s.ffill().tolist())
print("bfill()      :", s.bfill().tolist())
print("interpolate():", s.interpolate().tolist())

## 3.1 Measure what your filling did

In [ ]:
s = pd.Series([10., 20., np.nan, 40., 50.])

rows = []
for label, filled in [
    ("leave it",      s),
    ("fillna(mean)",  s.fillna(s.mean())),
    ("fillna(0)",     s.fillna(0)),
    ("interpolate()", s.interpolate()),
]:
    rows.append({"choice": label,
                 "mean": round(filled.mean(), 2),
                 "std":  round(filled.std(), 2),
                 "n":    filled.count()})

pd.DataFrame(rows).set_index("choice")

Three things to read off that table:

- **`fillna(0)` changed the mean** from 30.0 to 24.0. A missing temperature is not 0 degrees;
  a missing salary is not an unpaid employee. **Zero is a value**, and inserting it changes
  the answer.
- **`fillna(mean)` held the mean but shrank the spread**, 18.26 → 15.81. Every filled value
  sits exactly at the centre, so it adds no variation. The data now looks *more consistent*
  than it really is, and any model trained on it will be overconfident.
- **`interpolate()` guessed 30** — which happens to be exactly right here, because the series
  is a straight line. On real data it is a guess like any other.

### ⚠️ `ffill` assumes an order

In [ ]:
# Sorted by time - ffill is reasonable
readings = pd.DataFrame({
    "time":  ["09:00", "10:00", "11:00", "12:00"],
    "temp":  [21.0, 22.0, np.nan, 23.0],
})
print("by time:", readings["temp"].ffill().tolist(), " <- carries 22 forward, sensible")

# Sorted alphabetically by name - ffill copies a stranger's value
people = pd.DataFrame({
    "name":   ["Amit", "Neha", "Ravi", "Sara"],
    "salary": [48000, np.nan, 52000, 61000],
})
print("by name:", people["salary"].ffill().tolist(), " <- Neha just got Amit's salary")

# Carrying the previous value forward only makes sense if the rows are in a
# MEANINGFUL sequence. Sort first, or do not use it.

---
# 4. Duplicates

In [ ]:
dd = pd.DataFrame({
    "id":   [1, 2, 2, 3, 3],
    "name": ["A", "B", "B", "C", "C-diff"],
    "v":    [10, 20, 20, 30, 31],
})
print(dd)
print()
print("duplicated():", dd.duplicated().tolist())
print("drop_duplicates()            rows:", len(dd.drop_duplicates()))
print("drop_duplicates(subset='id') rows:", len(dd.drop_duplicates(subset="id")))

**`subset` changes the question.** Without it, a row counts as a duplicate only if *every*
column matches — so row 4 (`C-diff`, 31) survives. With `subset="id"`, id 3 appears twice,
so one goes, and **you have just silently chosen which version of C to keep.**

In [ ]:
# keep= decides WHICH copy survives
print("keep='first' :", dd.drop_duplicates(subset="id", keep="first")["name"].tolist())
print("keep='last'  :", dd.drop_duplicates(subset="id", keep="last")["name"].tolist())
print("keep=False   :", len(dd.drop_duplicates(subset="id", keep=False)), "rows (keeps none)")

In [ ]:
# LOOK at the conflicts before you drop them
conflicts = dd[dd.duplicated(subset="id", keep=False)]
print(conflicts)

# Two copies of id 3 that DISAGREE is a data problem worth investigating,
# not something to quietly resolve with keep="first".

## 4.1 ⚠️ Near-duplicates, and why order matters

In [ ]:
fz = pd.Series(["Pune", " pune", "PUNE", "Mumbai"])

print("nunique() as-is      :", fz.nunique())
print("after strip + lower  :", fz.str.strip().str.lower().nunique())

# drop_duplicates compares strings EXACTLY, so a leading space makes two
# identical cities look different.

In [ ]:
# And remember the Day 15 trap
s2 = pd.Series(["Sales", "sales", "IT", "IT"])
print("str.title():", s2.str.strip().str.title().unique().tolist(), " <- IT became It")
print("str.lower():", s2.str.strip().str.lower().unique().tolist(), " <- safe")

In [ ]:
# THE ORDER OF YOUR CLEANING STEPS CHANGES THE ANSWER
m = pd.DataFrame({"city": ["Pune", " Pune", "Delhi"], "v": [1, 1, 2]})

print("dedupe, then clean:", len(m.drop_duplicates()), "rows")

m2 = m.copy()
m2["city"] = m2["city"].str.strip()
print("clean, then dedupe:", len(m2.drop_duplicates()), "rows")

# Deduplicating first leaves "Pune" and " Pune" as separate rows, because at
# that moment they genuinely ARE different strings.
#
# NORMALISE THE TEXT FIRST, THEN DEDUPLICATE. This is a real ordering rule,
# not a style preference - the two pipelines give different data.

---
# 5. Putting it together — a pipeline that reports itself

In [ ]:
# A deliberately messy file
with open("staff.csv", "w") as f:
    f.write("emp_id,name,dept,salary\n")
    f.write("1,Ravi, Sales,52000\n")
    f.write("2,Sara,sales,-999\n")
    f.write("3,Amit,IT,48000\n")
    f.write("3,amit ,IT,49000\n")
    f.write("4,Neha,N/A,unknown\n")
    f.write("5,Kiran,IT,61000\n")

# ---- 1. read, declaring the disguises up front
df = pd.read_csv("staff.csv", na_values=["-999", "N/A", "unknown", ""])
start = len(df)
print("rows:", start)
print(df.isna().mean().round(2))

In [ ]:
# ---- 2. normalise text BEFORE deduplicating
for col in ["name", "dept"]:
    df[col] = df[col].str.strip().str.lower()

# ---- 3. now duplicates are findable - LOOK before dropping
dupes = df.duplicated(subset=["emp_id"], keep=False)
print("conflicting rows:")
print(df[dupes])
df = df.drop_duplicates(subset=["emp_id"], keep="last")

In [ ]:
# ---- 4. fill, and measure what it cost
before = df["salary"].agg(["mean", "std", "count"])
df["salary"] = df["salary"].fillna(df["salary"].median())
after = df["salary"].agg(["mean", "std", "count"])

print(pd.concat([before, after], axis=1, keys=["before", "after"]).round(2))
print()
print(f"{start} rows in, {len(df)} out")
print()
print(df)

The standard deviation fell from **6245 to 4550** — that is the price of filling two gaps
with the median, and anyone reading the output can see it.

Anyone reading this can reconstruct exactly what you did to the data. **That is the standard
to aim for.**

---
# 6. Recap — the twelve things to remember

1. There is no neutral choice — dropping and filling both cost.
2. `isna()` misses `-999`, `"N/A"` and `"unknown"`. Use `na_values`.
3. A numeric column stored as text means a sentinel is hiding.
4. Read the **proportion** missing, not just the count.
5. Ask **why** it is missing before deciding what to do.
6. `dropna()` removes a row for a gap in any column at all.
7. `dropna(subset=[...])` is almost always the better call.
8. `fillna(0)` on a measurement changes the mean. Avoid it.
9. `fillna(mean)` holds the mean but **shrinks the spread**.
10. `ffill` assumes an order — sort first, or do not use it.
11. `drop_duplicates`: `subset=` decides what counts, `keep=` which survives.
12. Normalise text **before** deduplicating. The order changes the answer.

---

### 📝 Now open **`Day19_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Take a messy CSV and write a cleaning pipeline that prints a receipt.
- Fill one column three different ways and compare the mean and std each time.
- Find a dataset where dropping the gaps biases the result, and say how.

### Next class — Topic 1.19: Outlier detection & treatment
The 1.5 × IQR rule, z-scores, and deciding whether a strange value is an error or the
finding.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*